# MediaStream: Google Colab Pipeline
This notebook is formatted to run in **Google Colab**. 

### ⚠️ Initial Setup
Before running, please upload the following 3 files to the Colab environment (the `/content/` folder):
1. `DLDATA.csv`
2. `movies.csv`
3. `ratings.csv`


## Step 1: Data Preprocessing
Loads survey data and movie data, encodes IDs, normalizes ratings, and creates a TF-IDF content matrix.


In [ ]:
"""
Step 1: Data Preprocessing
--------------------------
- Loads DLDATA.csv (survey data) and movies.csv + ratings.csv (from DLCASE)
- Renames long feature names to short, clean names
- Maps languages: English, Hindi, Telugu
- Encodes genres: Action, Romance, Thriller, Drama, Family, Sci-Fi, Horror, Comedy
- Builds a clean dataset ready for deep learning
"""
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

DLCASE = "."

def load_and_rename_survey():
    """Load DLDATA.csv and rename the ridiculously long column names."""
    df = pd.read_csv("DLDATA.csv")
    
    rename_map = {
        'user_id': 'user_id',
        'content_id': 'content_id',
        'Timestamp': 'timestamp',
        'Which genres do you enjoy the most? (Select up to 3)': 'preferred_genres',
        'What type of content do you prefer?': 'content_type',
        'When do you usually watch content? (Select all that apply)': 'watch_time',
        'What is your preferred episode length?': 'episode_length',
        'How do you usually find new content to watch?': 'discovery_method',
        'How often do you re-watch your favorite shows or movies?': 'rewatch_freq',
        'Which device do you primarily use for streaming?': 'device',
        'How satisfied are you with the current content recommendations?': 'satisfaction',
        'Based on your preferred content type, please share the name of a recent favorite you watched (e.g., a movie, TV show, or short film).': 'recent_favorite',
        'rating': 'rating'
    }
    
    df = df.rename(columns=rename_map)
    print(f"  Survey data loaded: {len(df)} users")
    print(f"  Columns after renaming: {list(df.columns)}")
    return df


def load_movies():
    """Load movies.csv from DLCASE."""
    path = os.path.join(DLCASE, "movies.csv")
    df = pd.read_csv(path)
    df = df.dropna(subset=['movieId', 'title'])
    df['movieId'] = df['movieId'].astype(int)
    
    # --- Language Mapping (ISO codes -> Full Names) ---
    lang_map = {
        'en': 'English', 'hi': 'Hindi', 'te': 'Telugu',
        'ta': 'Tamil', 'ml': 'Malayalam', 'kn': 'Kannada',
        'es': 'Spanish', 'fr': 'French', 'de': 'German',
        'ja': 'Japanese', 'ko': 'Korean', 'zh': 'Chinese',
        'cn': 'Chinese', 'ru': 'Russian', 'it': 'Italian',
        'pt': 'Portuguese', 'ar': 'Arabic', 'nl': 'Dutch',
        'sv': 'Swedish', 'no': 'Norwegian', 'da': 'Danish',
        'pl': 'Polish', 'ro': 'Romanian', 'hu': 'Hungarian',
        'cs': 'Czech', 'el': 'Greek', 'he': 'Hebrew',
        'th': 'Thai', 'vi': 'Vietnamese', 'uk': 'Ukrainian',
        'bg': 'Bulgarian', 'la': 'Latin', 'eo': 'Esperanto',
        'gd': 'Scottish Gaelic', 'ga': 'Irish', 'gl': 'Galician',
        'eu': 'Basque', 'si': 'Sinhala', 'bn': 'Bengali',
        'ur': 'Urdu', 'ne': 'Nepali', 'my': 'Burmese',
        'km': 'Khmer', 'bo': 'Tibetan', 'sq': 'Albanian',
        'sw': 'Swahili', 'yi': 'Yiddish', 'ja': 'Japanese',
    }
    
    def get_primary_language(lang_str):
        """Get the primary (first) language as full name."""
        if pd.isna(lang_str):
            return 'English'  # default
        parts = str(lang_str).split('|')
        first = parts[0].strip().lower()
        return lang_map.get(first, first.upper())
    
    def get_all_languages(lang_str):
        """Get all languages as full names."""
        if pd.isna(lang_str):
            return 'English'
        parts = str(lang_str).split('|')
        mapped = [lang_map.get(p.strip().lower(), p.strip().upper()) for p in parts]
        return '|'.join(mapped)
    
    df['primary_language'] = df['language_str'].apply(get_primary_language)
    df['all_languages'] = df['language_str'].apply(get_all_languages)
    
    # Clean genres
    df['genres_str'] = df['genres_str'].fillna('Unknown')
    df['genres_list'] = df['genres_str'].str.split('|')
    
    # Create combined content text for TF-IDF similarity
    # Language is repeated 3x to boost its weight in similarity scoring
    df['content_text'] = (
        df['genres_str'].str.replace('|', ' ', regex=False) + ' ' +
        df['primary_language'] + ' ' +
        df['primary_language'] + ' ' +
        df['primary_language']
    )
    
    print(f"  Movies loaded: {len(df)} movies")
    print(f"  Languages found: {df['primary_language'].nunique()} unique")
    print(f"  Top 3 languages: {df['primary_language'].value_counts().head(3).to_dict()}")
    return df


def load_ratings():
    """Load ratings.csv from DLCASE."""
    path = os.path.join(DLCASE, "ratings.csv")
    df = pd.read_csv(path)
    df = df.dropna(subset=['userId', 'movieId', 'rating'])
    df['movieId'] = df['movieId'].astype(int)
    print(f"  Ratings loaded: {len(df)} interactions")
    return df


def main():
    print("=" * 60)
    print("🚀 STEP 1: DATA PREPROCESSING")
    print("=" * 60)
    
    # --- Load all datasets ---
    print("\n[1/6] Loading survey data (DLDATA.csv)...")
    survey_df = load_and_rename_survey()
    
    print("\n[2/6] Loading movies catalog...")
    movies_df = load_movies()
    
    print("\n[3/6] Loading ratings/interactions...")
    ratings_df = load_ratings()
    
    # --- Filter ratings to valid movies only ---
    valid_ids = set(movies_df['movieId'].unique())
    ratings_df = ratings_df[ratings_df['movieId'].isin(valid_ids)].copy()
    print(f"  Valid ratings after filtering: {len(ratings_df)}")
    
    # --- Encode Users and Movies ---
    print("\n[4/6] Encoding users and movies for neural network...")
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()
    
    # Fit user encoder on ALL users from both survey and ratings
    all_users = list(set(
        list(ratings_df['userId'].unique()) + 
        list(survey_df['user_id'].unique())
    ))
    user_encoder.fit(all_users)
    ratings_df['user_enc'] = user_encoder.transform(ratings_df['userId'])
    
    # Fit movie encoder on ALL movies
    movie_encoder.fit(movies_df['movieId'])
    ratings_df['movie_enc'] = movie_encoder.transform(ratings_df['movieId'])
    movies_df['movie_enc'] = movie_encoder.transform(movies_df['movieId'])
    
    # Normalize ratings to 0-1 for sigmoid output
    r_min, r_max = ratings_df['rating'].min(), ratings_df['rating'].max()
    ratings_df['rating_norm'] = (ratings_df['rating'] - r_min) / (r_max - r_min + 1e-9)
    
    num_users = len(user_encoder.classes_)
    num_items = len(movie_encoder.classes_)
    print(f"  Encoded Users: {num_users}")
    print(f"  Encoded Movies: {num_items}")
    
    # --- Build TF-IDF Content Matrix ---
    print("\n[5/6] Building TF-IDF content similarity matrix...")
    # Sort movies by encoded ID so matrix index = encoded ID
    movies_df = movies_df.sort_values('movie_enc').reset_index(drop=True)
    
    tfidf = TfidfVectorizer(stop_words='english', max_features=2000)
    content_matrix = tfidf.fit_transform(movies_df['content_text'])
    print(f"  Content matrix shape: {content_matrix.shape}")
    
    # --- Save Everything ---
    print("\n[6/6] Saving all preprocessed data...")
    
    # Clean survey data
    survey_df.to_csv("cleaned_survey.csv", index=False)
    
    # Movies and ratings
    movies_df.to_csv("movies_clean.csv", index=False)
    ratings_df.to_csv("ratings_clean.csv", index=False)
    
    # Encoders
    with open("user_encoder.pkl", "wb") as f:
        pickle.dump(user_encoder, f)
    with open("movie_encoder.pkl", "wb") as f:
        pickle.dump(movie_encoder, f)
    
    # TF-IDF artifacts
    with open("tfidf_vectorizer.pkl", "wb") as f:
        pickle.dump(tfidf, f)
    save_npz("content_matrix.npz", content_matrix)
    
    # Save metadata
    meta = {
        'num_users': num_users,
        'num_items': num_items,
        'r_min': r_min,
        'r_max': r_max
    }
    with open("meta.pkl", "wb") as f:
        pickle.dump(meta, f)
    
    print("\n" + "=" * 60)
    print("✅ PREPROCESSING COMPLETE!")
    print("=" * 60)
    print("\nFiles saved:")
    print("  📄 cleaned_survey.csv    - Survey data with clean column names")
    print("  📄 movies_clean.csv      - Movies with language & genre tags")
    print("  📄 ratings_clean.csv     - User-movie interactions (encoded)")
    print("  📦 user_encoder.pkl      - User ID encoder")
    print("  📦 movie_encoder.pkl     - Movie ID encoder")
    print("  📦 tfidf_vectorizer.pkl  - TF-IDF model for content similarity")
    print("  📦 content_matrix.npz    - Precomputed content vectors")
    print("  📦 meta.pkl              - Metadata (counts, normalization)")


if __name__ == "__main__":
    main()


# Run preprocessing
if __name__ == "__main__":
    main()


## Step 2: NCF Model Architecture
Defines the Neural Collaborative Filtering deep learning model using TensorFlow/Keras.


In [ ]:
"""
Step 2: Deep Learning Model Architecture
-----------------------------------------
Neural Collaborative Filtering (NCF) with Genre & Language Embeddings.
This is a production-grade model that learns user-item interaction patterns.
"""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf


def build_ncf(num_users: int, num_items: int, embed_dim: int = 64) -> tf.keras.Model:
    """
    Neural Collaborative Filtering Model.
    
    Architecture:
        User Embedding (64-d) + Item Embedding (64-d)
        -> Concatenate (128-d)
        -> Dense 256 (ReLU + BN + Dropout)
        -> Dense 128 (ReLU + BN + Dropout)
        -> Dense 64  (ReLU + BN + Dropout)
        -> Dense 1   (Sigmoid -> predicted rating 0-1)
    """
    user_in = tf.keras.Input(shape=(1,), name='user_input')
    item_in = tf.keras.Input(shape=(1,), name='item_input')

    user_emb = tf.keras.layers.Embedding(
        num_users, embed_dim,
        embeddings_regularizer=tf.keras.regularizers.l2(1e-5),
        name='user_embedding'
    )(user_in)

    item_emb = tf.keras.layers.Embedding(
        num_items, embed_dim,
        embeddings_regularizer=tf.keras.regularizers.l2(1e-5),
        name='item_embedding'
    )(item_in)

    user_flat = tf.keras.layers.Flatten()(user_emb)
    item_flat = tf.keras.layers.Flatten()(item_emb)

    x = tf.keras.layers.Concatenate()([user_flat, item_flat])

    # Deep MLP
    x = tf.keras.layers.Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    x = tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    out = tf.keras.layers.Dense(1, activation='sigmoid', name='prediction')(x)

    model = tf.keras.Model(inputs=[user_in, item_in], outputs=out, name='NCF')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    return model


if __name__ == "__main__":
    m = build_ncf(100, 100)
    m.summary()



## Step 3: Model Training
Splits the data into 80/20 train/test, and trains the NCF model.


In [ ]:
"""
Step 3: Train the NCF Deep Learning Model
------------------------------------------
Trains on preprocessed ratings data with EarlyStopping and LR scheduling.
Run AFTER step1_preprocess.py
"""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from step2_model import build_ncf


def train():
    print("=" * 60)
    print("STEP 3: TRAINING NCF MODEL")
    print("=" * 60)

    # Load preprocessed data
    print("\n[1/4] Loading preprocessed data...")
    ratings = pd.read_csv("ratings_clean.csv")

    with open("meta.pkl", "rb") as f:
        meta = pickle.load(f)

    num_users = meta['num_users']
    num_items = meta['num_items']
    print(f"  Users: {num_users}, Items: {num_items}, Ratings: {len(ratings)}")

    # Prepare arrays
    X_user = ratings['user_enc'].values
    X_item = ratings['movie_enc'].values
    y = ratings['rating_norm'].values.astype(np.float32)

    # Train/Val split
    Xu_tr, Xu_te, Xi_tr, Xi_te, y_tr, y_te = train_test_split(
        X_user, X_item, y, test_size=0.2, random_state=42
    )
    print(f"  Train: {len(y_tr)}, Validation: {len(y_te)}")

    # Build model
    print("\n[2/4] Building NCF model...")
    model = build_ncf(num_users, num_items, embed_dim=64)
    model.summary()

    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, verbose=1
        )
    ]

    # Train
    print("\n[3/4] Training...")
    history = model.fit(
        [Xu_tr, Xi_tr], y_tr,
        validation_data=([Xu_te, Xi_te], y_te),
        epochs=30, batch_size=256,
        callbacks=callbacks, verbose=1
    )

    # Save
    print("\n[4/4] Saving trained model...")
    model.save("ncf_model.h5")

    # Print final metrics
    val_loss = min(history.history['val_loss'])
    val_mae = min(history.history['val_mae'])
    print(f"\n  Best Val Loss : {val_loss:.4f}")
    print(f"  Best Val MAE  : {val_mae:.4f}")

    print("\n" + "=" * 60)
    print("TRAINING COMPLETE! Model saved -> ncf_model.h5")
    print("=" * 60)


if __name__ == "__main__":
    train()


# Run training
if __name__ == "__main__":
    train()


## Step 4: Hybrid Recommendation Engine
Combines NCF deep learning, TF-IDF content similarity, and Language Boost.


In [ ]:
"""
Step 4: Hybrid Recommendation Engine
--------------------------------------
Combines:
  1. NCF Deep Learning scores (collaborative filtering)
  2. TF-IDF Content Similarity (genre + language matching)
  3. Language Boost (Telugu/Hindi/English priority from user history)

When a user logs in, the engine:
  - Fetches their watch history
  - Detects their preferred language & genres
  - Scores ALL unseen movies using the hybrid formula
  - Returns top-K personalized recommendations
"""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

DLCASE = "."


class RecommendationEngine:
    """Hybrid Deep Learning Recommendation Engine."""

    def __init__(self):
        print("Loading Recommendation Engine...")

        # Load metadata
        with open("meta.pkl", "rb") as f:
            self.meta = pickle.load(f)

        # Load encoders
        with open("user_encoder.pkl", "rb") as f:
            self.user_enc = pickle.load(f)
        with open("movie_encoder.pkl", "rb") as f:
            self.movie_enc = pickle.load(f)

        # Load data
        self.movies = pd.read_csv("movies_clean.csv")
        self.ratings = pd.read_csv("ratings_clean.csv")
        self.survey = pd.read_csv("cleaned_survey.csv")

        # Sort movies by encoded ID for index alignment
        self.movies = self.movies.sort_values('movie_enc').reset_index(drop=True)

        # Load NCF model
        self.ncf = tf.keras.models.load_model("ncf_model.h5", compile=False)

        # Load TF-IDF content matrix
        self.content_matrix = load_npz("content_matrix.npz")

        self.num_users = self.meta['num_users']
        self.num_items = self.meta['num_items']

        print(f"  Engine ready: {self.num_users} users, {self.num_items} movies")

    # ------------------------------------------------------------------
    def get_user_history(self, user_id: str) -> pd.DataFrame:
        """Get all movies a user has previously watched with details."""
        user_ratings = self.ratings[self.ratings['userId'] == user_id].copy()
        if user_ratings.empty:
            return pd.DataFrame()

        # Merge with movie info
        history = user_ratings.merge(
            self.movies[['movieId', 'title', 'genres_str', 'primary_language',
                         'popularity', 'movie_enc']],
            on='movieId', how='left'
        )
        # Sort by rating (highest first)
        history = history.sort_values('rating', ascending=False)
        return history

    # ------------------------------------------------------------------
    # Known Telugu/Hindi movie keywords for language detection from survey
    TELUGU_KEYWORDS = [
        'dhurandhar', 'salaar', 'salar', 'rrr', 'pushpa', 'animal', 'dhruva',
        'baahubali', 'bahubali', 'raja rani', 'just married', 'godavari',
        'couple friendly', 'kishkindapuri', 'with love', 'sirai', 'og',
        'splitsvilla', '8 vasantalu', 'isha', 'jigris', 'saaiyara',
        'rajsaab', 'raj saab', 'ustad bhagat', 'vikram', 'hit 3', 'hit3',
        'dude', 'laapata', 'made in korea', 'naruto', 'durandhar', 'dhurandar',
        'sarvam maya', 'hi nana', 'gandhi talks', 'om shanti',
    ]
    HINDI_KEYWORDS = [
        'pushpa', 'animal', 'laapata ladies', 'stranger things',
        'money heist', 'dhurandhar', 'durandhar',
    ]

    def _detect_language_from_survey(self, user_id: str) -> list:
        """Detect user's preferred language from survey recent_favorite field."""
        survey_row = self.survey[self.survey['user_id'] == user_id]
        if survey_row.empty:
            return []

        favorite = str(survey_row.iloc[0].get('recent_favorite', '')).lower().strip()
        if not favorite or favorite == 'nan':
            return []

        # Check for Telugu movie keywords
        for kw in self.TELUGU_KEYWORDS:
            if kw in favorite:
                return ['Telugu', 'Hindi', 'English']

        # Check for Hindi movie keywords
        for kw in self.HINDI_KEYWORDS:
            if kw in favorite:
                return ['Hindi', 'Telugu', 'English']

        return ['English']

    # ------------------------------------------------------------------
    def get_user_preferences(self, user_id: str) -> dict:
        """Detect user's preferred language and genres from watch history + survey."""
        history = self.get_user_history(user_id)
        prefs = {'languages': [], 'genres': [], 'avg_rating': 0}

        # Always check survey for genre preferences
        survey_row = self.survey[self.survey['user_id'] == user_id]
        if not survey_row.empty:
            genres_str = survey_row.iloc[0].get('preferred_genres', '')
            if pd.notna(genres_str):
                prefs['genres'] = [g.strip() for g in str(genres_str).replace('|', ',').split(',') if g.strip()]

        # Detect language from survey favorites (Telugu/Hindi detection)
        survey_langs = self._detect_language_from_survey(user_id)

        if history.empty:
            prefs['languages'] = survey_langs if survey_langs else ['English']
            return prefs

        # Extract preferred languages from watch history
        if 'primary_language' in history.columns:
            lang_counts = history.groupby('primary_language')['rating'].agg(['count', 'mean'])
            lang_counts['score'] = lang_counts['count'] * lang_counts['mean']
            history_langs = lang_counts.sort_values('score', ascending=False).index.tolist()
        else:
            history_langs = []

        # Merge: survey language takes priority (since survey shows real preference)
        if survey_langs:
            # Put survey-detected language first, then add history languages
            combined = list(survey_langs)
            for lang in history_langs:
                if lang not in combined:
                    combined.append(lang)
            prefs['languages'] = combined
        else:
            prefs['languages'] = history_langs

        # Extract preferred genres from history (supplement survey genres)
        all_genres = []
        for gs in history['genres_str'].dropna():
            all_genres.extend([g.strip() for g in str(gs).split('|')])
        if all_genres:
            from collections import Counter
            genre_counts = Counter(all_genres)
            history_genres = [g for g, _ in genre_counts.most_common(5)]
            # Merge with survey genres
            for g in history_genres:
                if g not in prefs['genres']:
                    prefs['genres'].append(g)

        prefs['avg_rating'] = history['rating'].mean()
        return prefs

    # ------------------------------------------------------------------
    def _ncf_scores(self, user_enc: int, movie_encs: np.ndarray) -> np.ndarray:
        """Get NCF predicted scores for a user across multiple movies."""
        user_arr = np.full(len(movie_encs), user_enc)
        preds = self.ncf.predict([user_arr, movie_encs], verbose=0, batch_size=1024)
        return preds.flatten()

    # ------------------------------------------------------------------
    def _content_scores(self, watched_encs: list, candidate_encs: np.ndarray) -> np.ndarray:
        """Compute content similarity between watched movies and candidates."""
        if not watched_encs:
            return np.zeros(len(candidate_encs))

        # Average the TF-IDF vectors of all watched movies to get a user profile
        watched_vecs = self.content_matrix[watched_encs]
        user_profile = np.asarray(watched_vecs.mean(axis=0))
        if user_profile.ndim == 1:
            user_profile = user_profile.reshape(1, -1)

        # Compute similarity to all candidate movies
        candidate_vecs = self.content_matrix[candidate_encs]
        sims = cosine_similarity(user_profile, candidate_vecs)
        return sims.flatten()

    # ------------------------------------------------------------------
    def _language_boost(self, preferred_langs: list, candidate_encs: np.ndarray) -> np.ndarray:
        """Strong language boost — heavily rewards matching language, penalizes mismatch."""
        if not preferred_langs:
            return np.zeros(len(candidate_encs))

        boost = np.zeros(len(candidate_encs))
        top_lang = preferred_langs[0]  # primary preferred language

        for i, enc in enumerate(candidate_encs):
            if enc < len(self.movies):
                movie_lang = self.movies.iloc[enc]['primary_language']
                if movie_lang == top_lang:
                    boost[i] = 1.0  # full boost for #1 language
                elif movie_lang in preferred_langs:
                    rank = preferred_langs.index(movie_lang)
                    boost[i] = max(0.7 - rank * 0.15, 0.2)
                else:
                    boost[i] = -0.3  # penalize non-preferred languages
        return boost

    # ------------------------------------------------------------------
    def _filter_by_language(self, candidate_encs: np.ndarray,
                            preferred_langs: list, min_results: int = 20) -> np.ndarray:
        """Filter candidates to preferred language. Falls back if too few results."""
        if not preferred_langs:
            return candidate_encs

        top_lang = preferred_langs[0]

        # Try: only top preferred language
        lang_mask = np.array([
            self.movies.iloc[enc]['primary_language'] == top_lang
            if enc < len(self.movies) else False
            for enc in candidate_encs
        ])
        filtered = candidate_encs[lang_mask]

        if len(filtered) >= min_results:
            return filtered

        # Fallback: all preferred languages
        lang_mask = np.array([
            self.movies.iloc[enc]['primary_language'] in preferred_langs
            if enc < len(self.movies) else False
            for enc in candidate_encs
        ])
        filtered = candidate_encs[lang_mask]

        if len(filtered) >= min_results:
            return filtered

        # Last fallback: return all candidates (not enough in preferred lang)
        return candidate_encs

    # ------------------------------------------------------------------
    def _filter_by_genre(self, candidate_encs: np.ndarray,
                         genre_filter: list) -> np.ndarray:
        """Strict genre filter — only keep movies that contain at least one selected genre."""
        if not genre_filter:
            return candidate_encs

        mask = np.array([
            any(g.strip() in str(self.movies.iloc[enc].get('genres_str', '')).split('|')
                for g in genre_filter)
            if enc < len(self.movies) else False
            for enc in candidate_encs
        ])
        filtered = candidate_encs[mask]
        return filtered if len(filtered) > 0 else candidate_encs

    # ------------------------------------------------------------------
    def recommend(self, user_id: str, top_k: int = 10, genre_filter: list = None, lang_filter: str = None) -> list:
        """
        Generate top-K hybrid recommendations for a user.

        Strategy: LANGUAGE-FIRST filtering, then hybrid scoring.
        1. Detect user's preferred language from watch history
        2. Filter candidates to that language
        3. Score using: 0.5 * NCF + 0.3 * Content + 0.2 * Language Boost

        Returns a list of dicts with movie info + scores.
        """
        # Check if user exists
        if user_id not in self.user_enc.classes_:
            return self._cold_start_recommend(user_id, top_k, genre_filter=genre_filter, lang_filter=lang_filter)

        user_enc_id = self.user_enc.transform([user_id])[0]

        # Get watched movie encoded IDs from ratings (which has movie_enc)
        user_ratings = self.ratings[self.ratings['userId'] == user_id]
        watched_encs = set(user_ratings['movie_enc'].dropna().astype(int).tolist()) if not user_ratings.empty else set()

        # Get user preferences
        prefs = self.get_user_preferences(user_id)

        # Candidate movies = all movies NOT yet watched
        all_encs = np.arange(self.num_items)
        candidate_mask = ~np.isin(all_encs, list(watched_encs))
        candidate_encs = all_encs[candidate_mask]

        if len(candidate_encs) == 0:
            return []

        # LANGUAGE-FIRST: if UI lang_filter provided, use it; otherwise use user prefs
        if lang_filter and lang_filter != 'Any':
            preferred_langs = [lang_filter]
        else:
            preferred_langs = prefs.get('languages', [])
        candidate_encs = self._filter_by_language(candidate_encs, preferred_langs,
                                                   min_results=top_k)

        # GENRE FILTER: strict — only show movies matching selected genres
        if genre_filter:
            candidate_encs = self._filter_by_genre(candidate_encs, genre_filter)

        if len(candidate_encs) == 0:
            return []

        # Score 1: NCF Deep Learning
        ncf_scores = self._ncf_scores(user_enc_id, candidate_encs)

        # Score 2: Content Similarity (genre + language via TF-IDF)
        content_scores = self._content_scores(list(watched_encs), candidate_encs)

        # Score 3: Language Boost (still applied for ranking within filtered set)
        lang_boost = self._language_boost(preferred_langs, candidate_encs)

        # Hybrid combination — language has strong influence
        hybrid_scores = (
            0.45 * ncf_scores +
            0.30 * content_scores +
            0.25 * lang_boost
        )

        # Get top-K indices
        top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

        # Build results
        results = []
        for idx in top_indices:
            enc_id = candidate_encs[idx]
            if enc_id >= len(self.movies):
                continue
            movie = self.movies.iloc[enc_id]
            results.append({
                'title': movie['title'],
                'year': movie.get('year', ''),
                'genres': movie.get('genres_str', 'Unknown'),
                'language': movie.get('primary_language', 'Unknown'),
                'popularity': float(movie.get('popularity', 0)),
                'ncf_score': float(ncf_scores[idx]),
                'content_score': float(content_scores[idx]),
                'lang_boost': float(lang_boost[idx]),
                'hybrid_score': float(hybrid_scores[idx]),
                'movie_enc': int(enc_id),
            })
        return results

    # ------------------------------------------------------------------
    def _cold_start_recommend(self, user_id: str, top_k: int,
                              genre_filter: list = None, lang_filter: str = None) -> list:
        """For new users not in training data, use survey preferences + popularity."""
        survey_row = self.survey[self.survey['user_id'] == user_id]
        preferred_genres = []
        if not survey_row.empty:
            gs = survey_row.iloc[0].get('preferred_genres', '')
            if pd.notna(gs):
                preferred_genres = [g.strip() for g in str(gs).replace('|', ',').split(',') if g.strip()]

        # If explicit genre filter provided, use that instead
        if genre_filter:
            preferred_genres = genre_filter

        # Score by genre match + popularity, FILTER by language FIRST
        results = []
        for _, movie in self.movies.iterrows():
            movie_lang = movie.get('primary_language', 'Unknown')

            # LANGUAGE FILTER: skip movies that don't match selected language
            if lang_filter and lang_filter != 'Any' and movie_lang != lang_filter:
                continue

            genres = str(movie.get('genres_str', '')).split('|')
            # Also handle comma-separated genres
            if len(genres) == 1 and ',' in genres[0]:
                genres = [g.strip() for g in genres[0].split(',')]

            genre_match = sum(1 for g in preferred_genres if g in genres)
            pop_score = float(movie.get('popularity', 0)) / 600.0
            score = 0.6 * (genre_match / max(len(preferred_genres), 1)) + 0.4 * min(pop_score, 1.0)
            results.append({
                'title': movie['title'],
                'year': movie.get('year', ''),
                'genres': movie.get('genres_str', 'Unknown'),
                'language': movie_lang,
                'popularity': float(movie.get('popularity', 0)),
                'ncf_score': 0.0,
                'content_score': score,
                'lang_boost': 0.0,
                'hybrid_score': score,
                'movie_enc': int(float(movie.get('movie_enc', 0))) if pd.notna(movie.get('movie_enc', 0)) else 0,
            })

        results.sort(key=lambda x: x['hybrid_score'], reverse=True)
        return results[:top_k]

    # ------------------------------------------------------------------
    def get_all_genres(self) -> list:
        """Get all unique genres from the catalog."""
        all_g = set()
        for gs in self.movies['genres_str'].dropna():
            for g in str(gs).split('|'):
                g = g.strip()
                if g and g != 'Unknown':
                    all_g.add(g)
        return sorted(all_g)

    def get_all_languages(self) -> list:
        """Get supported languages — Telugu, Hindi, English only."""
        return ['English', 'Hindi', 'Telugu']


if __name__ == "__main__":
    engine = RecommendationEngine()
    print("\nTesting with user U1...")
    history = engine.get_user_history("U1")
    print(f"Watch history: {len(history)} movies")
    prefs = engine.get_user_preferences("U1")
    print(f"Preferred languages: {prefs['languages'][:3]}")
    print(f"Preferred genres: {prefs['genres'][:5]}")
    recs = engine.recommend("U1", top_k=5)
    print(f"\nTop 5 recommendations:")
    for i, r in enumerate(recs, 1):
        print(f"  {i}. {r['title']} ({r['language']}) - {r['genres'][:40]} | Score: {r['hybrid_score']:.3f}")


# Test the engine
if __name__ == "__main__":
    engine = RecommendationEngine()
    print("\nTesting with user U1...")
    recs = engine.recommend("U1", top_k=5)
    for i, r in enumerate(recs, 1):
        print(f"  {i}. {r['title']} ({r['language']}) - {r['genres'][:40]} | Score: {r['hybrid_score']:.3f}")


## Step 5: Model Evaluation
Computes advanced metrics: MAE, RMSE, Precision, Hit Rate@10, and NDCG@10.


In [ ]:
"""
Model Evaluation & Architecture Report
-----------------------------------------
Shows all models, encoders/transformers used, and computes
accuracy metrics for the trained NCF hybrid recommendation system.
"""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import numpy as np
import pandas as pd
import pickle
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


def separator(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")


def main():
    print("\n" + "█"*70)
    print("█" + " "*68 + "█")
    print("█   MEDIASTREAM — COMPLETE MODEL EVALUATION REPORT" + " "*18 + "█")
    print("█" + " "*68 + "█")
    print("█"*70)

    # ─── 1. MODELS USED ────────────────────────────────────────────────
    separator("1. DEEP LEARNING MODEL — Neural Collaborative Filtering (NCF)")

    # Load metadata
    with open("meta.pkl", "rb") as f:
        meta = pickle.load(f)

    num_users = meta['num_users']
    num_items = meta['num_items']
    r_min = meta['r_min']
    r_max = meta['r_max']

    print(f"""
    Model Type      : Neural Collaborative Filtering (NCF)
    Framework       : TensorFlow / Keras {tf.__version__}
    Task            : Rating Prediction (Regression via Sigmoid)
    Loss Function   : Binary Cross-Entropy
    Optimizer       : Adam (lr=0.001)

    ┌─────────────────────────────────────────────────────────────┐
    │                    NCF ARCHITECTURE                         │
    ├─────────────────────────────────────────────────────────────┤
    │  Input: User ID (1,)        Input: Movie ID (1,)           │
    │         ↓                            ↓                     │
    │  User Embedding (64-d)      Item Embedding (64-d)          │
    │         ↓                            ↓                     │
    │         └──────── Concatenate ────────┘                     │
    │                      ↓ (128-d)                             │
    │              Dense 256 + ReLU                               │
    │              BatchNormalization                             │
    │              Dropout (0.30)                                 │
    │                      ↓                                     │
    │              Dense 128 + ReLU                               │
    │              BatchNormalization                             │
    │              Dropout (0.25)                                 │
    │                      ↓                                     │
    │              Dense 64 + ReLU                                │
    │              BatchNormalization                             │
    │              Dropout (0.20)                                 │
    │                      ↓                                     │
    │              Dense 1 + Sigmoid                              │
    │              (Predicted Rating 0–1)                         │
    └─────────────────────────────────────────────────────────────┘

    Regularization  : L2 (1e-5) on embeddings, Dropout, BatchNorm
    Callbacks       : EarlyStopping (patience=5), ReduceLROnPlateau
    Epochs          : Up to 30 (with early stopping)
    Batch Size      : 256
    Embedding Dim   : 64
    """)

    # Load and display model summary
    model = tf.keras.models.load_model("ncf_model.h5", compile=False)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )

    print("    Keras Model Summary:")
    print("    " + "─"*55)
    model.summary(print_fn=lambda x: print(f"    {x}"))

    total_params = model.count_params()
    print(f"\n    Total Trainable Parameters: {total_params:,}")

    # ─── 2. ENCODERS / TRANSFORMERS USED ────────────────────────────────
    separator("2. ENCODERS & TRANSFORMERS")

    # Label Encoders
    with open("user_encoder.pkl", "rb") as f:
        user_enc = pickle.load(f)
    with open("movie_encoder.pkl", "rb") as f:
        movie_enc = pickle.load(f)

    print(f"""
    ┌─────────────────────────────────────────────────────────────┐
    │              ENCODER / TRANSFORMER PIPELINE                 │
    ├──────────────────┬──────────────────────────────────────────┤
    │  Component       │  Details                                 │
    ├──────────────────┼──────────────────────────────────────────┤
    │  User Encoder    │  sklearn.preprocessing.LabelEncoder      │
    │                  │  Maps user IDs → integer indices          │
    │                  │  Num Classes: {num_users:<27}│
    ├──────────────────┼──────────────────────────────────────────┤
    │  Movie Encoder   │  sklearn.preprocessing.LabelEncoder      │
    │                  │  Maps movie IDs → integer indices         │
    │                  │  Num Classes: {num_items:<27}│
    ├──────────────────┼──────────────────────────────────────────┤
    │  TF-IDF          │  sklearn.feature_extraction.text          │
    │  Vectorizer      │      .TfidfVectorizer                    │
    │                  │  Converts genre+language text → vectors   │
    │                  │  Max Features: 2000                       │
    │                  │  Stop Words: english                      │
    ├──────────────────┼──────────────────────────────────────────┤
    │  Rating          │  Min-Max Normalization                    │
    │  Normalizer      │  rating_norm = (r - r_min)/(r_max-r_min) │
    │                  │  r_min={r_min}, r_max={r_max}                        │
    ├──────────────────┼──────────────────────────────────────────┤
    │  User Embedding  │  tf.keras.layers.Embedding               │
    │                  │  Input: {num_users} users → 64-d vectors          │
    │                  │  L2 regularization: 1e-5                  │
    ├──────────────────┼──────────────────────────────────────────┤
    │  Item Embedding  │  tf.keras.layers.Embedding               │
    │                  │  Input: {num_items} movies → 64-d vectors        │
    │                  │  L2 regularization: 1e-5                  │
    ├──────────────────┼──────────────────────────────────────────┤
    │  Cosine          │  sklearn.metrics.pairwise                 │
    │  Similarity      │      .cosine_similarity                  │
    │                  │  Content-based similarity scoring         │
    └──────────────────┴──────────────────────────────────────────┘
    """)

    # TF-IDF details
    with open("tfidf_vectorizer.pkl", "rb") as f:
        tfidf = pickle.load(f)

    content_matrix = load_npz("content_matrix.npz")
    vocab_size = len(tfidf.vocabulary_)
    print(f"    TF-IDF Vocabulary Size   : {vocab_size}")
    print(f"    Content Matrix Shape     : {content_matrix.shape}")
    print(f"    Content Matrix Non-zeros : {content_matrix.nnz:,}")
    print(f"    Sparsity                 : {(1 - content_matrix.nnz / (content_matrix.shape[0]*content_matrix.shape[1]))*100:.2f}%")

    # Show sample vocabulary
    top_terms = sorted(tfidf.vocabulary_.items(), key=lambda x: x[1])[:20]
    print(f"\n    Top TF-IDF Terms (sample): {', '.join([t[0] for t in top_terms])}")

    # ─── 3. HYBRID SCORING FORMULA ──────────────────────────────────────
    separator("3. HYBRID RECOMMENDATION FORMULA")

    print("""
    ┌─────────────────────────────────────────────────────────────┐
    │               HYBRID SCORING FORMULA                        │
    ├─────────────────────────────────────────────────────────────┤
    │                                                             │
    │   hybrid_score = 0.45 × NCF_score                          │
    │                + 0.30 × Content_similarity (TF-IDF)         │
    │                + 0.25 × Language_boost                      │
    │                                                             │
    ├─────────────────────────────────────────────────────────────┤
    │  NCF Score      : Deep Learning collaborative prediction    │
    │  Content Score  : Cosine similarity of TF-IDF vectors       │
    │  Language Boost : +1.0 primary lang, +0.55 secondary,       │
    │                   -0.3 penalty for non-preferred             │
    ├─────────────────────────────────────────────────────────────┤
    │  Cold Start     : 0.60 × Genre_match + 0.40 × Popularity   │
    └─────────────────────────────────────────────────────────────┘
    """)

    # ─── 4. ACCURACY METRICS ────────────────────────────────────────────
    separator("4. MODEL ACCURACY & EVALUATION METRICS")

    # Load ratings data
    ratings = pd.read_csv("ratings_clean.csv")
    print(f"\n    Total Ratings        : {len(ratings):,}")
    print(f"    Unique Users         : {ratings['userId'].nunique()}")
    print(f"    Unique Movies        : {ratings['movieId'].nunique()}")
    print(f"    Rating Range (norm)  : [{ratings['rating_norm'].min():.3f}, {ratings['rating_norm'].max():.3f}]")
    print(f"    Rating Range (raw)   : [{ratings['rating'].min()}, {ratings['rating'].max()}]")

    # Split data same way as training
    X_user = ratings['user_enc'].values
    X_item = ratings['movie_enc'].values
    y = ratings['rating_norm'].values.astype(np.float32)

    Xu_tr, Xu_te, Xi_tr, Xi_te, y_tr, y_te = train_test_split(
        X_user, X_item, y, test_size=0.2, random_state=42
    )

    print(f"\n    Train Set Size       : {len(y_tr):,}")
    print(f"    Test Set Size        : {len(y_te):,}")

    # ── 4a. Rating Prediction Metrics ──
    print(f"\n    {'─'*50}")
    print(f"    4a. RATING PREDICTION METRICS (on test set)")
    print(f"    {'─'*50}")

    # Predict on test set
    y_pred_test = model.predict([Xu_te, Xi_te], verbose=0, batch_size=1024).flatten()
    y_pred_train = model.predict([Xu_tr, Xi_tr], verbose=0, batch_size=1024).flatten()

    # Metrics on normalized scale (0-1)
    test_mae_norm = mean_absolute_error(y_te, y_pred_test)
    test_rmse_norm = np.sqrt(mean_squared_error(y_te, y_pred_test))
    train_mae_norm = mean_absolute_error(y_tr, y_pred_train)
    train_rmse_norm = np.sqrt(mean_squared_error(y_tr, y_pred_train))

    # Convert back to original scale
    scale = r_max - r_min
    test_mae_orig = test_mae_norm * scale
    test_rmse_orig = test_rmse_norm * scale
    train_mae_orig = train_mae_norm * scale
    train_rmse_orig = train_rmse_norm * scale

    # Evaluate using model.evaluate
    print("\n    Running model.evaluate() on test set...")
    test_loss, test_mae_eval, test_rmse_eval = model.evaluate(
        [Xu_te, Xi_te], y_te, verbose=0, batch_size=1024
    )
    train_loss, train_mae_eval, train_rmse_eval = model.evaluate(
        [Xu_tr, Xi_tr], y_tr, verbose=0, batch_size=1024
    )

    print(f"""
    ┌───────────────────────────────────────────────────────────────┐
    │            RATING PREDICTION ACCURACY                         │
    ├─────────────────────┬──────────────────┬──────────────────────┤
    │  Metric             │  Train Set       │  Test Set            │
    ├─────────────────────┼──────────────────┼──────────────────────┤
    │  Loss (BCE)         │  {train_loss:<16.6f}│  {test_loss:<20.6f}│
    │  MAE (normalized)   │  {train_mae_norm:<16.6f}│  {test_mae_norm:<20.6f}│
    │  RMSE (normalized)  │  {train_rmse_norm:<16.6f}│  {test_rmse_norm:<20.6f}│
    │  MAE (orig 1-5)     │  {train_mae_orig:<16.6f}│  {test_mae_orig:<20.6f}│
    │  RMSE (orig 1-5)    │  {test_rmse_orig:<16.6f}│  {test_rmse_orig:<20.6f}│
    └─────────────────────┴──────────────────┴──────────────────────┘
    """)

    # ── 4b. Classification / Ranking Metrics ──
    print(f"    {'─'*50}")
    print(f"    4b. RANKING / RECOMMENDATION METRICS (on test set)")
    print(f"    {'─'*50}")

    # Threshold: if predicted > 0.6 = "liked", actual > 0.6 = "liked"
    threshold = 0.6
    y_liked_actual = (y_te >= threshold).astype(int)
    y_liked_pred = (y_pred_test >= threshold).astype(int)

    tp = np.sum((y_liked_pred == 1) & (y_liked_actual == 1))
    fp = np.sum((y_liked_pred == 1) & (y_liked_actual == 0))
    fn = np.sum((y_liked_pred == 0) & (y_liked_actual == 1))
    tn = np.sum((y_liked_pred == 0) & (y_liked_actual == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(y_te)

    print(f"""
    Threshold for "liked" : >= {threshold} (normalized)  [= {threshold * scale + r_min:.1f}/5 original]

    ┌───────────────────────────────────────────────────────────────┐
    │             CLASSIFICATION METRICS                            │
    ├─────────────────────┬─────────────────────────────────────────┤
    │  Accuracy           │  {accuracy*100:<8.2f}%                          │
    │  Precision          │  {precision*100:<8.2f}%                          │
    │  Recall             │  {recall*100:<8.2f}%                          │
    │  F1-Score           │  {f1*100:<8.2f}%                          │
    ├─────────────────────┼─────────────────────────────────────────┤
    │  True Positives     │  {tp:<8,}                               │
    │  False Positives    │  {fp:<8,}                               │
    │  True Negatives     │  {tn:<8,}                               │
    │  False Negatives    │  {fn:<8,}                               │
    └─────────────────────┴─────────────────────────────────────────┘
    """)

    # ── 4c. Per-User Hit Rate & NDCG@K ──
    print(f"    {'─'*50}")
    print(f"    4c. TOP-K RECOMMENDATION METRICS")
    print(f"    {'─'*50}")

    K_VALUES = [5, 10]

    # Group test set by user
    test_df = pd.DataFrame({
        'user_enc': Xu_te, 'movie_enc': Xi_te,
        'actual': y_te, 'predicted': y_pred_test
    })

    for K in K_VALUES:
        hits = 0
        ndcg_scores = []
        total_users = 0

        for user_id, group in test_df.groupby('user_enc'):
            if len(group) < 2:
                continue
            total_users += 1

            # Actual top-K items (by true rating)
            actual_top = set(group.nlargest(K, 'actual')['movie_enc'].values)

            # Predicted top-K items
            pred_top = group.nlargest(K, 'predicted')['movie_enc'].values

            # Hit Rate: does at least 1 actual liked item appear in predicted top-K?
            hit = len(set(pred_top) & actual_top) > 0
            if hit:
                hits += 1

            # NDCG@K
            relevance = [1.0 if m in actual_top else 0.0 for m in pred_top]
            dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevance))
            ideal = sorted(relevance, reverse=True)
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcg_scores.append(ndcg)

        hit_rate = hits / total_users if total_users > 0 else 0
        avg_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0

        header = f"TOP-{K} METRICS  (evaluated on {total_users} users)"
        pad = 62 - len(header) - 4
        hr_label = f"Hit Rate@{K}"
        ndcg_label = f"NDCG@{K}"
        print(f"""
    +---------------------------------------------------------------+
    |  {header}{' '*pad}|
    +---------------------+-----------------------------------------+
    |  {hr_label:<19} |  {hit_rate*100:<8.2f}%                          |
    |  {ndcg_label:<19} |  {avg_ndcg:<8.4f}                           |
    +---------------------+-----------------------------------------+""")

    # ─── 5. DATASET STATISTICS ──────────────────────────────────────────
    separator("5. DATASET STATISTICS")

    movies = pd.read_csv("movies_clean.csv")
    survey = pd.read_csv("cleaned_survey.csv")

    print(f"""
    ┌───────────────────────────────────────────────────────────────┐
    │                    DATASET OVERVIEW                            │
    ├─────────────────────┬─────────────────────────────────────────┤
    │  Survey Responses   │  {len(survey):<8,}                               │
    │  Total Movies       │  {len(movies):<8,}                               │
    │  Total Ratings      │  {len(ratings):<8,}                               │
    │  Unique Users       │  {ratings['userId'].nunique():<8,}                               │
    │  Unique Languages   │  {movies['primary_language'].nunique():<8,}                               │
    ├─────────────────────┼─────────────────────────────────────────┤
    │  Avg Rating (raw)   │  {ratings['rating'].mean():<8.2f}                               │
    │  Ratings Std Dev    │  {ratings['rating'].std():<8.2f}                               │
    │  Sparsity           │  {(1 - len(ratings)/(ratings['userId'].nunique()*ratings['movieId'].nunique()))*100:<8.2f}%                          │
    └─────────────────────┴─────────────────────────────────────────┘
    """)

    # Language distribution
    print("    Language Distribution in Catalog:")
    lang_dist = movies['primary_language'].value_counts().head(10)
    for lang, count in lang_dist.items():
        pct = count / len(movies) * 100
        bar = "█" * int(pct / 2)
        print(f"      {lang:<15} {count:>6,}  ({pct:5.1f}%)  {bar}")

    # Genre distribution
    print("\n    Top Genre Distribution in Catalog:")
    all_genres = []
    for gs in movies['genres_str'].dropna():
        all_genres.extend([g.strip() for g in str(gs).split('|') if g.strip() and g.strip() != 'Unknown'])
    from collections import Counter
    genre_counts = Counter(all_genres).most_common(10)
    for genre, count in genre_counts:
        pct = count / len(movies) * 100
        bar = "█" * int(pct / 2)
        print(f"      {genre:<15} {count:>6,}  ({pct:5.1f}%)  {bar}")

    # ─── 6. COMPLETE PIPELINE SUMMARY ───────────────────────────────────
    separator("6. COMPLETE PIPELINE SUMMARY")

    print("""
    ┌─────────────────────────────────────────────────────────────┐
    │                 END-TO-END PIPELINE                          │
    ├─────────────────────────────────────────────────────────────┤
    │                                                             │
    │  Step 1: Data Preprocessing (step1_preprocess.py)           │
    │    ├── Load DLDATA.csv (survey), movies.csv, ratings.csv    │
    │    ├── LabelEncoder → user IDs & movie IDs to integers      │
    │    ├── Min-Max Normalization → ratings to [0, 1]            │
    │    ├── TfidfVectorizer → genre+language text to vectors     │
    │    └── Save: encoders, TF-IDF, content matrix, clean CSVs   │
    │                                                             │
    │  Step 2: Model Architecture (step2_model.py)                │
    │    └── NCF: Embedding → Concat → Dense(256,128,64) → σ      │
    │                                                             │
    │  Step 3: Training (step3_train.py)                          │
    │    ├── 80/20 train/test split                               │
    │    ├── Adam optimizer, Binary Cross-Entropy loss            │
    │    ├── EarlyStopping + ReduceLROnPlateau                    │
    │    └── Save: ncf_model.h5                                   │
    │                                                             │
    │  Step 4: Hybrid Engine (step4_engine.py)                    │
    │    ├── NCF scores (collaborative filtering)                 │
    │    ├── TF-IDF Cosine Similarity (content-based)             │
    │    ├── Language Boost (survey + history-aware)               │
    │    └── Hybrid: 0.45×NCF + 0.30×Content + 0.25×LangBoost    │
    │                                                             │
    │  Step 5: Streamlit Dashboard (step5_app.py)                 │
    │    ├── Glassmorphism UI with animations                     │
    │    ├── User profile, watch history, preferences             │
    │    ├── Cold start support for new users                     │
    │    └── Explainable AI (reason for each recommendation)      │
    │                                                             │
    └─────────────────────────────────────────────────────────────┘
    """)

    print("█"*70)
    print("█  EVALUATION COMPLETE                                              █")
    print("█"*70 + "\n")


if __name__ == "__main__":
    main()


# Run evaluation
if __name__ == "__main__":
    main()
